# 06_networks — Build temporal bipartite networks

> **Environment:** requires the project venv **`.venv311`** (Python 3.11) as the Jupyter kernel — the pipeline dependencies are installed only there. `run_all.command` uses it automatically. See `README.md` → Environment setup.

**Input:** All files in `data/output/edges/edges_*.jsonl`  
**Output:** 
- `data/output/networks/G_{window}.graphml` — primary (Gephi, Cytoscape, yEd, igraph compatible)
- `data/output/networks/G_{window}.gexf` — Gephi-preferred (preserves more attribute typing)
- `data/output/networks/G_{window}.html` — pyvis interactive preview (open in any browser, no install)
- `data/output/networks/G_actors_{window}.graphml` — actor–actor projection (derived view; + pyvis preview)

Per README §6: one weighted bipartite graph per window. All whitelisted actors + all concept clusters are added as nodes upfront — isolated nodes (an actor with zero edges in this window) are meaningful absences and kept intentionally.

Actor–concept edges carry a `polarity` attribute from `CONCEPT_POLARITY` (the polarity of the concept invoked — **not** sentence sentiment; see the caveat below). Each bipartite graph also gets a derived actor–actor projection.

## Pipeline steps in this notebook

1. Setup & paths
2. Load all edge files
3. Build one bipartite graph per window
4. Compute node statistics (degree, total weight) — used for sizing in viz
5. Validation report (isolated nodes, total weight, expected-spike check)
6. Save GraphML (primary format)
7. Save GEXF (Gephi-preferred)
8. Save pyvis interactive HTML — edges coloured by concept polarity, with legend
9. Actor–actor projection — derived 'actor congruence' view (Leifeld & Haunss 2012)

## Step 1: Setup & paths

In [ ]:
import json
import sys
from pathlib import Path
from collections import defaultdict, Counter

import networkx as nx

_cwd = Path().resolve()
ROOT = next(
    (p for p in [_cwd] + list(_cwd.parents) if (p / 'src').is_dir()),
    _cwd,
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.alias_map   import ACTOR_WHITELIST, ACTOR_TYPE
from src.concept_dict import CONCEPT_DICT, CONCEPT_POLARITY, POLARITY_COLOR

EDGES_DIR   = ROOT / 'data' / 'output' / 'edges'
NETWORKS_DIR = ROOT / 'data' / 'output' / 'networks'
NETWORKS_DIR.mkdir(parents=True, exist_ok=True)

from src.time_windows import window_file_label
# Typed + per-format output layout; filenames carry the window's chronological
# index (01_buildup_mar ... 08_aftermath_aug) so they sort chronologically.
for _sub in ('cooccurrence/graphml', 'cooccurrence/gexf', 'cooccurrence/html',
             'actor_actor/graphml', 'actor_actor/html'):
    (NETWORKS_DIR / _sub).mkdir(parents=True, exist_ok=True)

print(f'Edges dir   : {EDGES_DIR}')
print(f'Networks dir: {NETWORKS_DIR}')
print(f'Actors in whitelist : {len(ACTOR_WHITELIST)}')
print(f'Concept clusters    : {len(CONCEPT_DICT)}')

## Step 2: Load edge files

In [ ]:
edge_files = sorted(EDGES_DIR.glob('edges_*.jsonl'))
print(f'Found {len(edge_files)} edge file(s):')
for f in edge_files:
    print(f'  {f.name}')
assert edge_files, 'No edge files found — run 05_edges first'

edges_by_window = defaultdict(list)
for f in edge_files:
    with open(f, encoding='utf-8') as fh:
        for line in fh:
            e = json.loads(line)
            edges_by_window[e['window']].append(e)

print(f'\nEdges loaded per window:')
for w in sorted(edges_by_window):
    print(f'  {w:20s}  {len(edges_by_window[w])} edges')

## Step 3: Build bipartite graphs

Each graph has every whitelisted actor and every concept as a node, even if zero edges. 
Isolated nodes are intentional: 'actor X has no co-occurrence with any concept in this window' is a real finding. 

Node attributes:
- `node_type` ∈ {'actor', 'concept'} — used by viz tools for shape/color
- `bipartite` ∈ {0, 1}                — NetworkX bipartite convention
- `degree`                            — edge count in this window
- `total_weight`                      — sum of edge weights (for sizing)
- `total_weight_norm`                 — sum of normalized weights (for cross-window sizing)

Edge attributes:
- `weight`            — raw co-occurrence count
- `weight_normalized` — per-article
- `n_sources`         — distinct outlets contributing
- `top_source`        — single largest contributor

In [ ]:
graphs = {}

for window, edges in sorted(edges_by_window.items()):
    G = nx.Graph()
    G.graph['window'] = window

    # Nodes — all whitelisted actors + all concepts (some will be isolated)
    for actor in sorted(ACTOR_WHITELIST):
        G.add_node(actor, bipartite=0, node_type='actor',
                   entity_type=ACTOR_TYPE.get(actor, 'UNKNOWN'))
    for concept in CONCEPT_DICT.keys():
        G.add_node(concept, bipartite=1, node_type='concept',
                   entity_type='CONCEPT')

    # Edges with full attributes
    for e in edges:
        sources = e.get('sources', {}) or {}
        top_src = max(sources.items(), key=lambda kv: kv[1])[0] if sources else ''
        G.add_edge(
            e['actor'], e['concept'],
            weight=int(e['weight']),
            weight_normalized=float(e['weight_normalized']),
            n_sources=int(len(sources)),
            top_source=top_src,
            polarity=CONCEPT_POLARITY[e['concept']],
        )

    # Node-level stats for sizing in viz
    for n in G.nodes():
        G.nodes[n]['degree']           = G.degree(n)
        G.nodes[n]['total_weight']     = sum(d['weight'] for _, _, d in G.edges(n, data=True))
        G.nodes[n]['total_weight_norm'] = round(
            sum(d['weight_normalized'] for _, _, d in G.edges(n, data=True)), 4
        )

    graphs[window] = G

print(f'Built {len(graphs)} bipartite graph(s)')

### Edge polarity is a property of the concept, not sentence sentiment

Each actor–concept edge now carries a `polarity` attribute taken directly from `CONCEPT_POLARITY[concept]` — diplomacy = **positive**, nuclear_program / strike_claims = **neutral**, deterrence / military_action = **negative**.

This is the polarity of the **concept being invoked**, not a sentence-level sentiment or stance classification. It does **not** detect negation or speaker stance: an actor *rejecting escalation* still produces a negative-cluster (`deterrence`) edge, because the deterrence concept appeared in the sentence. True sentence-level polarity (negation/stance aware) is future work (see Part B).

## Step 4: Validation report

Per README: print isolated nodes, total weight, and flag isolated whitelist actors in climax windows.

In [ ]:
CLIMAX_WINDOWS = {'climax_w1', 'climax_w2'}

for window, G in graphs.items():
    actor_nodes   = [n for n, d in G.nodes(data=True) if d['node_type'] == 'actor']
    concept_nodes = [n for n, d in G.nodes(data=True) if d['node_type'] == 'concept']
    actors_connected = [n for n in actor_nodes  if G.degree(n) > 0]
    actors_isolated  = [n for n in actor_nodes  if G.degree(n) == 0]
    concepts_connected = [n for n in concept_nodes if G.degree(n) > 0]
    total_weight = sum(d['weight'] for _, _, d in G.edges(data=True))

    print(f'\n========== {window} ==========')
    print(f'  Actors connected   : {len(actors_connected)}/{len(actor_nodes)}')
    print(f'  Actors isolated    : {len(actors_isolated)}  → {sorted(actors_isolated)}')
    print(f'  Concepts connected : {len(concepts_connected)}/{len(concept_nodes)}')
    print(f'  Total edges        : {G.number_of_edges()}')
    print(f'  Total edge weight  : {total_weight}')

    if window in CLIMAX_WINDOWS and actors_isolated:
        critical = [a for a in actors_isolated if a in {'USA', 'IRAN', 'ISRAEL', 'TRUMP', 'KHAMENEI'}]
        if critical:
            print(f'  WARNING: critical actors isolated in climax window: {critical}')

    # Top 5 strongest edges in this window
    top_edges = sorted(G.edges(data=True), key=lambda x: -x[2]['weight'])[:5]
    print(f'  Top 5 edges (raw weight):')
    for u, v, d in top_edges:
        print(f'    {d["weight"]:5d}  [{d["polarity"]:8s}]  {u} — {v}  (norm={d["weight_normalized"]:.3f})')

## Step 5: Save GraphML (primary format)

GraphML is the methodology's required format and is the lingua franca for graph tools: Gephi, Cytoscape, yEd, Graphia, igraph all read it natively.

In [ ]:
for window, G in graphs.items():
    out = NETWORKS_DIR / 'cooccurrence' / 'graphml' / f'{window_file_label(window)}_cooc.graphml'
    nx.write_graphml(G, out)
    print(f'GraphML : {out}')

## Step 6: Save GEXF (Gephi preferred)

GEXF preserves richer attribute typing than GraphML and is Gephi's native format. Recommended if you'll use Gephi as your primary visualizer.

In [ ]:
for window, G in graphs.items():
    out = NETWORKS_DIR / 'cooccurrence' / 'gexf' / f'{window_file_label(window)}_cooc.gexf'
    nx.write_gexf(G, out)
    print(f'GEXF    : {out}')

## Step 7: Pyvis interactive HTML (OPTIONAL)

Generates a self-contained HTML file you can open in any browser — no install, no Gephi launch, no learning curve. Useful for quick preview and for sharing with your supervisor.

Requires `pip install pyvis`. Skipped silently if not installed.

In [ ]:
try:
    from pyvis.network import Network
    HAS_PYVIS = True
except ImportError:
    HAS_PYVIS = False
    print('pyvis not installed — skip this cell or run: pip install pyvis')

# Node styling is a single two-way actor/concept distinction only. entity_type
# (GPE/PERSON/ORG/CONCEPT) is NOT encoded here — too many categories to stay
# readable; it lives only as GraphML metadata. Edges are coloured by concept
# polarity via POLARITY_COLOR.
ACTOR_COLOR   = '#4c72b0'   # every actor node: blue dot
CONCEPT_COLOR = '#8e44ad'   # every concept node: purple diamond

def polarity_legend_html(items):
    """Floating HTML legend (pyvis has no native legend support)."""
    rows = ''.join(
        f'<div style="display:flex;align-items:center;margin:2px 0;">'
        f'<span style="width:14px;height:14px;background:{color};display:inline-block;'
        f'margin-right:6px;border-radius:2px;"></span><span>{label}</span></div>'
        for label, color in items
    )
    return (
        '<div style="position:fixed;top:12px;right:12px;z-index:999;'
        'background:rgba(255,255,255,0.95);border:1px solid #ccc;border-radius:6px;'
        'padding:8px 10px;font-family:sans-serif;font-size:12px;'
        'box-shadow:0 1px 4px rgba(0,0,0,0.2);">'
        '<div style="font-weight:bold;margin-bottom:4px;">Edge polarity</div>'
        + rows + '</div>'
    )

def inject_legend(html_path, items):
    """Inject the floating legend div into a saved pyvis HTML file."""
    html = html_path.read_text(encoding='utf-8')
    html_path.write_text(
        html.replace('</body>', polarity_legend_html(items) + '</body>', 1),
        encoding='utf-8',
    )

POLARITY_LEGEND = [('positive', POLARITY_COLOR['positive']),
                   ('neutral',  POLARITY_COLOR['neutral']),
                   ('negative', POLARITY_COLOR['negative'])]

OPTIONS = '{"physics": {"forceAtlas2Based": {"gravitationalConstant": -60, "springLength": 140, "avoidOverlap": 0.8}, "solver": "forceAtlas2Based", "minVelocity": 0.75, "timestep": 0.4}, "interaction": {"hover": true, "navigationButtons": true, "keyboard": true}}'

if HAS_PYVIS:
    for window, G in graphs.items():
        net = Network(height='720px', width='100%', bgcolor='#ffffff',
                      font_color='#222222', notebook=False, directed=False,
                      cdn_resources='in_line')
        max_node_w = max((G.nodes[n]['total_weight'] for n in G.nodes()), default=1) or 1
        max_edge_w = max((d['weight'] for _, _, d in G.edges(data=True)), default=1) or 1

        for n, d in G.nodes(data=True):
            is_actor = d['node_type'] == 'actor'
            size = 10 + 35 * (d['total_weight'] / max_node_w) if d['degree'] else 8
            net.add_node(
                n, label=n,
                color=ACTOR_COLOR if is_actor else CONCEPT_COLOR,
                shape='dot' if is_actor else 'diamond',
                size=size,
                title=(f"{n}<br>node_type: {d['node_type']}<br>"
                       f"entity_type: {d.get('entity_type', '')}<br>"
                       f"degree: {d['degree']}<br>total weight: {d['total_weight']}"),
                physics=True,
            )
        for u, v, d in G.edges(data=True):
            pol = d.get('polarity', 'neutral')
            net.add_edge(
                u, v, value=d['weight'],
                width=1 + 6 * (d['weight'] / max_edge_w),
                color=POLARITY_COLOR.get(pol, '#9AA0A6'),
                title=(f"{u} — {v}<br>polarity: {pol}<br>weight: {d['weight']}<br>"
                       f"weight_normalized: {d['weight_normalized']}<br>"
                       f"top source: {d['top_source']}"),
            )
        net.set_options(OPTIONS)
        out = NETWORKS_DIR / 'cooccurrence' / 'html' / f'{window_file_label(window)}_cooc.html'
        net.write_html(str(out), open_browser=False, notebook=False)
        inject_legend(out, POLARITY_LEGEND)
        print(f'HTML    : {out.name}')

    print('\nBipartite previews written — edge colour = concept polarity; '
          'actor nodes = blue dots, concept nodes = purple diamonds.')

## Step 8: Actor–actor projection (derived view)

For each bipartite `G_t` we also save its **weighted actor–actor projection**: two actors are connected when they attach to a common concept, and the edge weight = number of shared concept neighbours. This is the **actor congruence network** of Leifeld & Haunss (2012).

It is a **secondary, derived view** for coalition-structure inspection only — the bipartite actor–concept network remains primary. The projection is lossy (the shared concept is no longer a node), so instead of a single edge colour each projected edge stores a *polarity profile*: `shared_concepts`, the `polarity_positive` / `polarity_neutral` / `polarity_negative` counts, and `dominant_polarity` (argmax of the three; ties → `mixed`). Projected edges are coloured by `dominant_polarity`.

Saved as `data/output/networks/G_actors_{window}.graphml` (+ a pyvis preview). `shared_concepts` is stored as a comma-joined string because GraphML edge attributes must be scalar.

In [ ]:
from networkx.algorithms import bipartite

projections = {}
for window, G in graphs.items():
    actor_nodes = {n for n, d in G.nodes(data=True) if d['bipartite'] == 0}
    G_actors = bipartite.weighted_projected_graph(G, actor_nodes)

    # Polarity profile per projected edge — the shared concept is no longer a
    # node, so record which concepts two actors share and their polarity mix.
    for u, v, d in G_actors.edges(data=True):
        shared = sorted(set(G[u]) & set(G[v]))            # common concept neighbours
        counts = Counter(CONCEPT_POLARITY[c] for c in shared)
        pos, neu, neg = counts['positive'], counts['neutral'], counts['negative']
        mx  = max(pos, neu, neg)
        top = [name for name, cnt in
               (('positive', pos), ('neutral', neu), ('negative', neg)) if cnt == mx]
        d['shared_concepts']   = ','.join(shared)         # GraphML stores scalars only
        d['n_shared_concepts'] = len(shared)
        d['polarity_positive'] = pos
        d['polarity_neutral']  = neu
        d['polarity_negative'] = neg
        d['dominant_polarity'] = top[0] if len(top) == 1 else 'mixed'

    out = NETWORKS_DIR / 'actor_actor' / 'graphml' / f'{window_file_label(window)}_actors.graphml'
    nx.write_graphml(G_actors, out)
    projections[window] = G_actors
    n_mixed = sum(1 for _, _, d in G_actors.edges(data=True) if d['dominant_polarity'] == 'mixed')
    print(f'projection : {out.name}  '
          f'({G_actors.number_of_edges()} actor-actor edges, {n_mixed} mixed)')

# pyvis previews of the projection — edges coloured by dominant_polarity
if HAS_PYVIS:
    PROJ_LEGEND = POLARITY_LEGEND + [('mixed', POLARITY_COLOR['mixed'])]
    OPTIONS_P = '{"physics": {"forceAtlas2Based": {"gravitationalConstant": -80, "springLength": 120, "avoidOverlap": 0.9}, "solver": "forceAtlas2Based", "minVelocity": 0.75, "timestep": 0.4}, "interaction": {"hover": true, "navigationButtons": true, "keyboard": true}}'
    for window, G_actors in projections.items():
        net = Network(height='720px', width='100%', bgcolor='#ffffff',
                      font_color='#222222', notebook=False, directed=False,
                      cdn_resources='in_line')
        max_w = max((d['weight'] for _, _, d in G_actors.edges(data=True)), default=1) or 1
        for n, d in G_actors.nodes(data=True):
            deg = G_actors.degree(n)
            net.add_node(n, label=n, color=ACTOR_COLOR, shape='dot',
                         size=10 + 3 * deg,
                         title=(f"{n}<br>entity_type: {d.get('entity_type', '')}<br>"
                                f"projected degree: {deg}"),
                         physics=True)
        for u, v, d in G_actors.edges(data=True):
            net.add_edge(u, v, value=d['weight'],
                         width=1 + 6 * (d['weight'] / max_w),
                         color=POLARITY_COLOR.get(d['dominant_polarity'], '#9AA0A6'),
                         title=(f"{u} — {v}<br>shared ({d['n_shared_concepts']}): {d['shared_concepts']}<br>"
                                f"dominant: {d['dominant_polarity']}<br>"
                                f"pos {d['polarity_positive']} / neu {d['polarity_neutral']} / neg {d['polarity_negative']}"))
        net.set_options(OPTIONS_P)
        out = NETWORKS_DIR / 'actor_actor' / 'html' / f'{window_file_label(window)}_actors.html'
        net.write_html(str(out), open_browser=False, notebook=False)
        inject_legend(out, PROJ_LEGEND)
        print(f'projection HTML : {out.name}')

print(f'\nActor-actor projections saved for {len(projections)} window(s) — '
      'derived "actor congruence" view (Leifeld & Haunss 2012); bipartite remains primary.')